In [9]:
import cv2
import time
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

options = vision.GestureRecognizerOptions(
    base_options=python.BaseOptions(model_asset_path="gesture_recognizer.task"),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
)
recognizer = vision.GestureRecognizer.create_from_options(options)

cap = cv2.VideoCapture(1)
start_time = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    timestamp = int((time.time() - start_time) * 1000)
    result = recognizer.recognize_for_video(mp_image, timestamp)

    text = "No hand"
    if result.gestures:
        best = result.gestures[0][0]
        text = best.category_name + "  " + str(round(best.score, 2))
    cv2.putText(frame, text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]
        height, width, _ = frame.shape

        for number, point in enumerate(hand):
            x = int(point.x * width)
            y = int(point.y * height)
            cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)
            cv2.putText(frame, str(number), (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        tip = hand[8]
        tip_x = int(tip.x * width)
        tip_y = int(tip.y * height)
        cv2.circle(frame, (tip_x, tip_y), 12, (0, 0, 255), -1)

    cv2.imshow("My webcam", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()